In [ ]:
from autocvd import autocvd
autocvd(num_gpus = 1)

backend = "torch"
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
if "KERAS_BACKEND" not in os.environ:
    os.environ["KERAS_BACKEND"] = backend

import bayesflow as bf
import numpy as np


INFO:bayesflow:Using backend 'torch'
When using torch backend, we need to disable autograd by default to avoid excessive memory usage. Use

with torch.enable_grad():
    ...

in contexts where you need gradients (e.g. custom training loops).
/export/home/vgiusepp/miniconda3/envs/bf/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
n_training = 100             #size of training set
n_trials = 20               #size of the order invariant input of the SetTransformer is (n_trials, 1)
n_test = 13                 #size of test set
n_subject_test = 5          #N_compositional_conditions
n_samples_posterior = 24


def score_log_norm(x, m, s):
    return -(x-m) / s**2

def simulate_(mu, sigma, n_subjects=1, n_trials=1):
    if isinstance(mu, (float, int)):
        mu = np.ones((n_subjects,)) * mu
        sigma = np.ones((n_subjects,)) * sigma
    data = np.zeros((n_subjects, n_trials, 1))
    for j_subject in range(n_subjects):
        for i_trial in range(n_trials):
            data[j_subject, i_trial] = np.random.normal(loc=mu[j_subject], scale=sigma[j_subject])
    if n_subjects == 1 and n_trials == 1:
        data = data[0, 0]
    elif n_subjects == 1:
        data = data[0]
    elif n_trials == 1:
        data = data[:, 0]
    return dict(sim_data=data)

def sample_hierarchical_priors(n_subjects=1):

    #Group level
    mean_mu = np.random.normal(-10, 10)
    mean_sigma = np.random.normal(5, 0,1)

    #Subject level
    mu = np.random.normal(loc=mean_mu, scale=1.0, size=n_subjects)
    sigma = np.random.normal(loc=mean_sigma, scale=0.1, size=n_subjects)
    return dict(mean_mu=mean_mu, 
                mean_sigma=mean_sigma, 
                mu=mu, 
                sigma=sigma)


def prior_global_score(x: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
    mean_mu = x["mean_mu"]
    mean_sigma = x["mean_sigma"]
    parts = {
        "mean_mu": score_log_norm(mean_mu, m=0.0, s=10.0),
        "mean_sigma": score_log_norm(mean_sigma, m=5.0, s=1.0),
    }
    return parts


param_names_global = ["mean_mu", "mean_sigma"]
adapter = (
    bf.adapters.Adapter()
    .to_array()
    .convert_dtype("float64", "float32")
    .concatenate(param_names_global, into="inference_variables")
    .rename("sim_data", "summary_variables")
)
workflow_global = bf.BasicWorkflow(
    adapter=adapter,
    summary_network=bf.networks.SetTransformer(summary_dim=16, dropout=0.1),
    inference_network=bf.networks.CompositionalDiffusionModel(),
)


simulator_hierarchical = bf.simulators.make_simulator([sample_hierarchical_priors, simulate_])
training_data = simulator_hierarchical.sample((n_training), n_trials=n_trials)
for i in range(n_training):
    data_to_save = {k:training_data[k][i] for k in training_data.keys()}
    np.savez(f"./data/training_data_{i}.npz", **data_to_save)

In [ ]:
a = np.load('./data/training_data_0.npz')
a_dict = load_npz_as_dict('./data/training_data_0.npz')
a_dict

{'mean_mu': array([-25.65668376]),
 'mean_sigma': array([5.]),
 'mu': array([-27.38521733]),
 'sigma': array([4.99296113]),
 'sim_data': array([[-28.93337271],
        [-29.33920475],
        [-26.93954204],
        [-33.01644475],
        [-24.45632929],
        [-36.81326816],
        [-30.01939418],
        [-25.39271057],
        [-22.41497703],
        [-30.07073126],
        [-27.67893618],
        [-25.66085897],
        [-31.55850571],
        [-25.95120408],
        [-33.05254784],
        [-24.63642737],
        [-29.38690069],
        [-29.48677044],
        [-27.27158792],
        [-41.90200424]])}

In [ ]:
def load_npz_as_dict(file):
        with np.load(file) as data:
            data = {k: v for k, v in data.items() if k in param_names_global or k in ['sim_data_projected']}
            return data
        
stream_data = np.load('/export/home/vgiusepp/diffusion-experiments/case_study5/project_stream/data/streams/data/simulation_0.npz')
stream_data_dict = load_npz_as_dict('/export/home/vgiusepp/diffusion-experiments/case_study5/project_stream/data/streams/data/simulation_0.npz')
stream_data_dict

param_names_global = ['m_Triaxial_halo', 'r_Triaxial_halo', 'q2_Triaxial_halo', 'rho_thin_disk', 'hr_thin_disk', 'hz_thin_disk', 'rho_thick_disk', 'hr_thick_disk', 'hz_thick_disk']
sim_data = "sim_data_projected"
adapter = (
    bf.adapters.Adapter()
    .to_array()
    .convert_dtype("float64", "float32")
    .concatenate(param_names_global, into="inference_variables")
    .rename("sim_data", "summary_variables")
)
workflow_global = bf.BasicWorkflow(
    adapter=adapter,
    summary_network=bf.networks.SetTransformer(summary_dim=16, dropout=0.1),
    inference_network=bf.networks.CompositionalDiffusionModel(),
)




{'sim_data_carthesian': array([[-7.61395454e+00, -4.86200762e+00,  4.80883300e-01,
          1.34663971e+02, -2.87827759e+02,  1.55375000e+02],
        [-7.67034197e+00, -3.40089178e+00,  1.39395630e+00,
          4.17544037e+02, -1.78278580e+02,  1.63418747e+02],
        [-7.66614437e+00, -4.71436644e+00,  6.28757477e-01,
          1.59514511e+02, -2.72993103e+02,  1.73661301e+02],
        ...,
        [-6.62175512e+00, -6.08287334e+00,  1.91552371e-01,
          1.34129974e+02, -3.11725220e+02,  1.38655594e+02],
        [-7.31938791e+00, -5.05105209e+00,  8.08884859e-01,
          3.07983917e+02, -1.73993378e+02,  1.42637817e+02],
        [-7.00004005e+00, -5.65044451e+00,  2.11607322e-01,
          1.12046135e+02, -3.19419922e+02,  1.39193085e+02]],
       shape=(3000, 6), dtype=float32),
 'sim_data_projected': array([[150.08382105, -48.31092113,   4.91007921,   4.74306549,
           1.5782469 , 554.63345792],
        [163.66499777, -35.20829263,   3.69534762,  16.61448795,
       

In [4]:

#let's create a random mask, the SetTrasnformer will be trained with a variable number of permutationally invariant observation
min_valid_fraction = 0.5    #at least 50% of the observation are valid
max_valid_fraction = 1.0    #at most 100% of the observation are valid
attention_mask_training = np.zeros((n_training, n_trials), dtype=np.float32)
for i in range(n_training):
    # Random fraction of valid particles for this sample
    valid_fraction = np.random.uniform(min_valid_fraction, max_valid_fraction)
    n_valid = int(n_trials * valid_fraction)
    # Randomly select which particles are valid
    valid_indices = np.random.choice(n_trials, size=n_valid, replace=False)
    attention_mask_training[i, valid_indices] = 1.0

def load_npz_as_dict(file):
    with np.load(file) as data:
        return dict(data)

history = workflow_global.fit_disk(
        root='./data/',
        pattern='*.npz',
        load_fn = load_npz_as_dict,
        epochs=2,
        batch_size=3,
        verbose=2,
        kwargs={'attention_mask': attention_mask_training},
    )

#TESTING USING MORE SUBJECTS=N_compositional_conditions
test_data = simulator_hierarchical.sample(n_test, n_subjects=n_subject_test, n_trials=n_trials, )
print('test data shape', test_data['sim_data'].shape)

#same as for the training set, we are going to use a mask for the SetTransformer
# Create mask with shape (N_TEST, N_SUBJECTS, N_PARTICLES) first
attention_mask_3d = np.zeros((n_test, n_subject_test, n_trials), dtype=np.float32)

for i in range(n_test):
    for j in range(n_subject_test):
        valid_fraction = np.random.uniform(min_valid_fraction, max_valid_fraction)
        n_valid = int(n_trials * valid_fraction)
        valid_indices = np.random.choice(n_trials, size=n_valid, replace=False)
        attention_mask_3d[i, j, valid_indices] = 1.0
# Flatten to (N_TEST * N_SUBJECTS, N_PARTICLES). The sim_data is flattened internally so we need to do the same for the attention mask
attention_mask_test = attention_mask_3d.reshape(n_test * n_subject_test, n_trials)

global_posterior = workflow_global.compositional_sample(
    num_samples=n_samples_posterior,
    conditions={'sim_data': test_data['sim_data']},
    compute_prior_score=prior_global_score,
    compositional_bridge_d1=1/n_subject_test,
    mini_batch_size=n_subject_test,
    method='two_step_adaptive',
    steps='adaptive',
    max_steps=1000,
    kwargs={'attention_mask': attention_mask_test},
)
print('Global posterior samples:')
print('mean_mu:', global_posterior['mean_mu'].shape)
print('mean_sigma:', global_posterior['mean_sigma'].shape)

INFO:bayesflow:Fitting on dataset instance of DiskDataset.
INFO:bayesflow:Building on a test batch.


Epoch 1/2
34/34 - 4s - 130ms/step - loss: 0.5459
Epoch 2/2
34/34 - 4s - 128ms/step - loss: 0.4761


INFO:bayesflow:Training completed in 9.86 seconds.


test data shape (13, 5, 20, 1)


Compositional Sampling:   0%|          | 0/1 [00:00<?, ?batch/s]


ValueError: In a nested call() argument, you cannot mix tensors and non-tensors. Received invalid mixed argument: kwargs={'compositional_bridge_d1': 0.2, 'mini_batch_size': 5, 'method': 'two_step_adaptive', 'steps': 'adaptive', 'max_steps': 1000, 'kwargs': {'attention_mask': tensor([[1., 0., 1.,  ..., 1., 1., 1.],
        [1., 1., 0.,  ..., 1., 1., 1.],
        [1., 1., 0.,  ..., 1., 0., 0.],
        ...,
        [1., 1., 0.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 0., 0., 1.]], device='cuda:0')}}

In [4]:
# Possible step in the right direction to avoid mixing tensor and non tensor kwargs as suggest by @Jonas aruda
workflow_global.approximator.inference_network.integrate_kwargs.update({
'method': 'two_step_adaptive',
'steps': 'adaptive',
'compositional_bridge_d1': 1/n_subject_test,
'mini_batch_size': n_subject_test,
})

global_posterior = workflow_global.compositional_sample(
    num_samples=n_samples_posterior,
    conditions={'sim_data': test_data['sim_data']},
    compute_prior_score=prior_global_score,
    attention_mask =attention_mask_test,
)


ValueError: Exception encountered when calling SetTransformer.call().

[1mInput 0 with name 'None' of layer 'dense_17' is incompatible with the layer: expected axis -1 of input shape to have value 64, but received input with shape (65, 4160)[0m

Arguments received by SetTransformer.call():
  • input_set=torch.Tensor(shape=torch.Size([65, 20, 1]), dtype=float32)
  • training=False
  • kwargs={'attention_mask': 'torch.Tensor(shape=torch.Size([65, 20]), dtype=float32)'}

In [4]:
attention_mask_test.shape

(65, 20)

In [ ]:
#Works for non compositional sampling
test_data['sim_data'] = test_data['sim_data'].reshape(n_test * n_subject_test, n_trials, 1)
global_posterior = workflow_global.sample(
    num_samples=n_samples_posterior,
    conditions={'sim_data': test_data['sim_data']},
    kwargs={'attention_mask': attention_mask_test},
    )

Sampling:   0%|          | 0/1 [00:00<?, ?batch/s]


ValueError: Exception encountered when calling SetTransformer.call().

[1mInput 0 with name 'None' of layer 'dense_8' is incompatible with the layer: expected axis -1 of input shape to have value 64, but received input with shape (65, 4160)[0m

Arguments received by SetTransformer.call():
  • input_set=torch.Tensor(shape=torch.Size([65, 20, 1]), dtype=float32)
  • training=False
  • kwargs={'attention_mask': 'torch.Tensor(shape=torch.Size([65, 20]), dtype=float32)'}